In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable
import uuid
from datetime import datetime, timezone

In [0]:
spark.sql("use catalog neogen")

In [0]:
spark.sql("create schema if not exists bronze_schema")

In [0]:
spark.sql("""create table if not exists neogen.bronze_schema.ingestion_control (
    layer string,
    table_name string,
    ts_col string,
    pk_col string,
    last_successful_ts timestamp,
    last_successful_pk bigint,
    last_run_id string,
    rows_written bigint,
    run_status string,
    updated_at timestamp
)using delta""")

In [0]:
tables_config = {
    "orders"   : {"pk_col" : "order_id" , "ts_col" : "updated_at"},
    "products" : { "pk_col" : "product_id", "ts_col" : "updated_at"},
    "payments" : { "pk_col" : "payment_id", "ts_col" : "processed_at"}
}
bronze_run_id=str(uuid.uuid4())
print(bronze_run_id)

In [0]:
def get_last_successful_watermark(table_name: str):
    ctrl = (
        spark.table("neogen.bronze_schema.ingestion_control")
        .filter(
            (f.col("layer") == 'bronze') &
            (f.col("table_name") == table_name) &
            (f.col("run_status") == "success")
        )
        .orderBy(f.col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()
    if not rows:
        return None, None

    return rows[0]["last_successful_ts"], rows[0]["last_successful_pk"]

In [0]:
def upsert_bronze_control(spark, table_name, ts_col, pk_col, last_ts, last_pk, rows_written,run_id):
    # 1. Create control DataFrame with run details
    control_df = spark.createDataFrame(
        [( "bronze",
            table_name,
            ts_col,
            pk_col,
            last_ts,
            int(last_pk) if last_pk is not None else None,
            run_id,
            int(rows_written),
            "success",
            datetime.now(timezone.utc)
        )],
        schema="""
            layer string,
            table_name string,
            ts_col string,
            pk_col string,
            last_successful_ts timestamp,
            last_successful_pk bigint,
            last_run_id string,
            rows_written bigint,
            run_status string,
            updated_at timestamp
        """
    )

    # 2. Merge control DataFrame into Delta target table
    dt = DeltaTable.forName(spark, "neogen.bronze_schema.ingestion_control")
    (dt.alias("t")
     .merge(control_df.alias("s"), "t.table_name = s.table_name AND t.layer = s.layer")
     .whenMatchedUpdate(set={
         "ts_col" : "s.ts_col",
         "pk_col" : "s.pk_col",
         "last_successful_ts" : "s.last_successful_ts",
         "last_successful_pk" : "s.last_successful_pk",
         "last_run_id" : "s.last_run_id",
         "rows_written" : "s.rows_written",
         "run_status" : "s.run_status",
         "updated_at" : "s.updated_at"
     })
     .whenNotMatchedInsertAll()
     .execute()
    )

In [0]:
def process_bronze_ingestion(tables_config, bronze_run_id):
    for table_name, cfg in tables_config.items():
        pk_col = cfg["pk_col"]
        ts_col = cfg["ts_col"]
        source_table = f"`neon-postgress-conn_catalog`.public.{table_name}"
        target_table = f"neogen.bronze_schema.{table_name}_raw"

        last_successful_ts, last_successful_pk = get_last_successful_watermark(table_name)
        print(f"\n=== Processing {table_name} ===")
        print("last_successful_ts =", last_successful_ts)
        print("last_successful_pk =", last_successful_pk)

        source_df = spark.read.table(source_table).withColumn(ts_col, f.col(ts_col).cast("timestamp"))

        if last_successful_ts is None:
            rows_to_load = source_df
        else:
            rows_to_load = source_df.filter(
                (f.col(ts_col) > f.lit(last_successful_ts)) |
                (
                    (f.col(ts_col) == f.lit(last_successful_ts)) &
                    (f.col(pk_col).cast("long") > f.lit(int(last_successful_pk)))
                )
            )

        rows_to_load = (
            rows_to_load
            .withColumn("bronze_ingested_at", f.current_timestamp())
            .withColumn("bronze_run_id", f.lit(bronze_run_id))
            .withColumn("bronze_source_table", f.lit(source_table))
        )

        row_count = rows_to_load.count()

        print(f"{table_name} rows_to_load = {row_count}")

        if row_count == 0:
            print(f"No new rows for {table_name}.")
            upsert_bronze_control(
                spark,
                table_name,
                ts_col,
                pk_col,
                last_successful_ts,
                last_successful_pk,
                row_count,
                bronze_run_id
            )
            continue

        rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)

        max_ts = rows_to_load.agg(f.max(ts_col).alias("max_ts")).collect()[0]["max_ts"]

        max_pk = (
            rows_to_load
            .filter(f.col(ts_col) == f.lit(max_ts))
            .agg(f.max(f.col(pk_col).cast("long")).alias("max_pk"))
            .collect()[0]["max_pk"]
        )

        upsert_bronze_control(spark, table_name, ts_col, pk_col, max_ts, max_pk, row_count, bronze_run_id)
        print(f"Wrote {row_count} rows to {target_table}")

#process_bronze_ingestion(tables_config, bronze_run_id)


In [0]:
def process_single_table(table_name, cfg, bronze_run_id):
    """Process a single table for bronze ingestion"""
    try:
        pk_col = cfg["pk_col"]
        ts_col = cfg["ts_col"]
        source_table = f"`neon-postgress-conn_catalog`.public.{table_name}"
        target_table = f"neogen.bronze_schema.{table_name}_raw"

        last_successful_ts, last_successful_pk = get_last_successful_watermark(table_name)
        print(f"\n=== Processing {table_name} ===")
        print(f"last_successful_ts = {last_successful_ts}")
        print(f"last_successful_pk = {last_successful_pk}")

        source_df = spark.read.table(source_table).withColumn(ts_col, f.col(ts_col).cast("timestamp"))

        if last_successful_ts is None:
            rows_to_load = source_df
        else:
            rows_to_load = source_df.filter(
                (f.col(ts_col) > f.lit(last_successful_ts)) |
                (
                    (f.col(ts_col) == f.lit(last_successful_ts)) &
                    (f.col(pk_col).cast("long") > f.lit(int(last_successful_pk)))
                )
            )

        rows_to_load = (
            rows_to_load
            .withColumn("bronze_ingested_at", f.current_timestamp())
            .withColumn("bronze_run_id", f.lit(bronze_run_id))
            .withColumn("bronze_source_table", f.lit(source_table))
        )

        row_count = rows_to_load.count()
        print(f"{table_name} rows_to_load = {row_count}")

        if row_count == 0:
            print(f"No new rows for {table_name}.")
            upsert_bronze_control(
                spark,
                table_name,
                ts_col,
                pk_col,
                last_successful_ts,
                last_successful_pk,
                row_count,
                bronze_run_id
            )
            return f"{table_name}: No new rows"

        rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)

        max_ts = rows_to_load.agg(f.max(ts_col).alias("max_ts")).collect()[0]["max_ts"]
        max_pk = (
            rows_to_load
            .filter(f.col(ts_col) == f.lit(max_ts))
            .agg(f.max(f.col(pk_col).cast("long")).alias("max_pk"))
            .collect()[0]["max_pk"]
        )

        upsert_bronze_control(spark, table_name, ts_col, pk_col, max_ts, max_pk, row_count, bronze_run_id)
        print(f"Wrote {row_count} rows to {target_table}")
        return f"{table_name}: Successfully loaded {row_count} rows"
    
    except Exception as e:
        print(f"Error processing {table_name}: {str(e)}")
        raise




In [0]:
# Execute in parallel using ThreadPoolExecutor
from concurrent.futures import ThreadPoolExecutor, as_completed
with ThreadPoolExecutor(max_workers=len(tables_config)) as executor:
    futures = {executor.submit(process_single_table, table_name, cfg, bronze_run_id): table_name 
                for table_name, cfg in tables_config.items()}
    
    for future in as_completed(futures):
        table_name = futures[future]
        try:
            result = future.result()
            print(f"Completed: {result}")
        except Exception as e:
            print(f"Failed {table_name}: {str(e)}")

In [0]:
%sql
select * from neogen.bronze_schema.ingestion_control